# Phase 2 — Spark ML
## Stock Movement Prediction Using S&P 500 Historical Data

**Task:** Apply 3 machine learning algorithms using PySpark MLlib to address the 3 problem statements from Phase 1.

| # | Problem | Algorithm | Evaluation Metrics |
|---|---------|-----------|-------------------|
| 1 | Classification — Stock direction | Random Forest | Accuracy, F1, AUC-ROC |
| 2 | Regression — Daily return | Gradient-Boosted Trees | RMSE, MAE, R² |
| 3 | Clustering — Stock grouping | K-Means | Silhouette Score |


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("SP500_Phase2_SparkML")
    .config("spark.driver.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/15 21:03:50 WARN Utils: Your hostname, MacBook-Air-114.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/04/15 21:03:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 21:03:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [2]:
from pathlib import Path

HDFS_PATH = "hdfs:///user/project/cleaned/cleaned_sp500.csv"

# Phase 1 writes this file; Jupyter cwd is usually the project folder
_project = Path.cwd().resolve()
_local_candidates = [
    _project / "cleaned_sp500.csv",
    _project.parent / "cleaned_sp500.csv",
]
local_csv = next((p for p in _local_candidates if p.is_file()), None)

df = None
try:
    df = spark.read.csv(HDFS_PATH, header=True, inferSchema=True)
    print(f"Loaded from HDFS: {HDFS_PATH}")
except Exception as e:
    if local_csv is None:
        raise FileNotFoundError(
            "No cleaned dataset found. Add cleaned_sp500.csv next to this notebook "
            "(run phase1_task4_5.ipynb to generate it from the Kaggle CSV + sp500_companies.csv), "
            "or use HDFS in your cluster environment.\n"
            f"Tried: {[str(p) for p in _local_candidates]}"
        ) from e
    df = spark.read.csv(str(local_csv), header=True, inferSchema=True)
    print(f"Loaded from local file: {local_csv}")

print(f"Rows: {df.count():,}  Columns: {len(df.columns)}")
df.printSchema()
df.show(5, truncate=False)


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/lorenzo/Downloads/hdshinde_lorenzop_qiangwu2_phase_1/cleaned_sp500.csv. SQLSTATE: 42K03

## Feature Engineering

Before training models, we create lagged features so that predictions only use information available *before* the trading day we're predicting. Specifically:

1. **Lagged daily return** — previous day's percentage return
2. **Lagged price range** — previous day's high minus low
3. **Lagged log-volume** — previous day's log-transformed volume
4. **Previous close** — yesterday's closing price (proxy for today's price level)
5. **Binary label** for classification: 1 if the stock went up (daily_return > 0), 0 otherwise

We also drop the first row per stock since it has no lag available.


In [ ]:
w = Window.partitionBy("symbol").orderBy("date")

df = df.withColumn("prev_daily_return", F.lag("daily_return", 1).over(w))
df = df.withColumn("prev_price_range", F.lag("price_range", 1).over(w))
df = df.withColumn("prev_log_volume", F.lag("log_volume", 1).over(w))
df = df.withColumn("prev_close", F.lag("close", 1).over(w))

# Binary label: 1 if stock went up, 0 if down/flat
df = df.withColumn("label", F.when(F.col("daily_return") > 0, 1.0).otherwise(0.0))

# Drop rows where lag is null (first trading day per stock)
df = df.na.drop(subset=["prev_daily_return", "prev_price_range", "prev_log_volume", "prev_close"])

print(f"Dataset after feature engineering: {df.count():,} rows")
print()
print("Label distribution (classification target):")
df.groupBy("label").count().orderBy("label").show()


In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Training set: {train_df.count():,} rows")
print(f"Test set:     {test_df.count():,} rows")


---
## Problem 1: Classification — Predict Stock Direction

**Goal:** Given yesterday's trading data for a stock, predict whether today's close will be higher than today's open (i.e., will the stock go up or down?).

### Why Random Forest?

Our Phase 1 EDA showed that daily returns have fat tails (kurtosis ~1.1) and are not normally distributed. Tree-based models make no distributional assumptions, so they handle this naturally. Random Forest specifically:

- Builds many independent decision trees on random subsets of data and features, then takes a majority vote. This reduces overfitting compared to a single tree.
- Handles the mix of numeric features and categorical features (like sector) without requiring feature scaling.
- Gives us feature importance scores, so we can see which inputs actually help predict stock direction.

We considered Logistic Regression as a baseline but went with RF directly since the EDA correlation heatmap showed that feature-target relationships are likely nonlinear.

**Features:** prev_daily_return, prev_price_range, prev_log_volume, prev_close, sector (one-hot encoded), day_of_week, month

**Metrics:** Accuracy, F1-score (weighted), AUC-ROC


In [ ]:
# --- Pipeline components ---
sector_indexer = StringIndexer(inputCol="sector", outputCol="sector_idx", handleInvalid="keep")
sector_encoder = OneHotEncoder(inputCol="sector_idx", outputCol="sector_vec")

clf_feature_cols = [
    "prev_daily_return", "prev_price_range", "prev_log_volume",
    "prev_close", "day_of_week", "month"
]
assembler_clf = VectorAssembler(
    inputCols=clf_feature_cols + ["sector_vec"],
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=100, maxDepth=10, seed=42
)

pipeline_clf = Pipeline(stages=[sector_indexer, sector_encoder, assembler_clf, rf])

# --- Train ---
print("Training Random Forest (numTrees=100, maxDepth=10)...")
model_clf = pipeline_clf.fit(train_df)
pred_clf = model_clf.transform(test_df)

# --- Evaluate ---
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

accuracy = acc_eval.evaluate(pred_clf)
f1 = f1_eval.evaluate(pred_clf)
auc_roc = auc_eval.evaluate(pred_clf)

print()
print("=== Random Forest Baseline Results ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc_roc:.4f}")


In [ ]:
# Grid search over numTrees and maxDepth using 3-fold cross-validation
paramGrid_clf = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

crossval_clf = CrossValidator(
    estimator=pipeline_clf,
    estimatorParamMaps=paramGrid_clf,
    evaluator=f1_eval,
    numFolds=3,
    seed=42,
    parallelism=2
)

print("Running 3-fold CV with 9 parameter combinations (27 model fits)...")
print("This may take several minutes.")
cv_model_clf = crossval_clf.fit(train_df)

# Evaluate best model on test set
best_pred_clf = cv_model_clf.transform(test_df)
best_accuracy = acc_eval.evaluate(best_pred_clf)
best_f1 = f1_eval.evaluate(best_pred_clf)
best_auc = auc_eval.evaluate(best_pred_clf)

# Extract best hyperparameters
best_rf = cv_model_clf.bestModel.stages[-1]
print()
print("=== Best Model After Hyperparameter Tuning ===")
print(f"numTrees:  {best_rf.getNumTrees}")
print(f"maxDepth:  {best_rf.getOrDefault('maxDepth')}")
print()
print(f"Accuracy:  {best_accuracy:.4f}  (baseline: {accuracy:.4f})")
print(f"F1-Score:  {best_f1:.4f}  (baseline: {f1:.4f})")
print(f"AUC-ROC:   {best_auc:.4f}  (baseline: {auc_roc:.4f})")

# Show all CV average scores
print()
print("=== Cross-Validation F1 Scores (averaged over 3 folds) ===")
cv_scores = cv_model_clf.avgMetrics
for i, params in enumerate(paramGrid_clf):
    nt = params[rf.numTrees]
    md = params[rf.maxDepth]
    print(f"  numTrees={nt:3d}, maxDepth={md:2d}  ->  F1 = {cv_scores[i]:.4f}")


In [ ]:
# Collect predictions to pandas for plotting
clf_pd = best_pred_clf.select("label", "prediction").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion Matrix ---
cm = pd.crosstab(clf_pd["label"], clf_pd["prediction"],
                 rownames=["Actual"], colnames=["Predicted"])
im = axes[0].imshow([[cm.iloc[0, 0], cm.iloc[0, 1]],
                      [cm.iloc[1, 0], cm.iloc[1, 1]]],
                     cmap="Blues")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Down (0)", "Up (1)"])
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["Down (0)", "Up (1)"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix", fontsize=13)
for i in range(2):
    for j in range(2):
        val = cm.iloc[i, j]
        axes[0].text(j, i, f"{val:,}", ha="center", va="center",
                     fontsize=14, color="white" if val > cm.values.max()/2 else "black")

# --- Feature Importance ---
importances = best_rf.featureImportances.toArray()
n_numeric = len(clf_feature_cols)
numeric_imps = importances[:n_numeric]
sector_imp_total = importances[n_numeric:].sum()

feat_names = clf_feature_cols + ["sector (combined)"]
feat_imps = list(numeric_imps) + [sector_imp_total]

imp_df = pd.DataFrame({"feature": feat_names, "importance": feat_imps})
imp_df = imp_df.sort_values("importance", ascending=True)

axes[1].barh(imp_df["feature"], imp_df["importance"], color="steelblue", edgecolor="black")
axes[1].set_title("Feature Importances", fontsize=13)
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

# Print classification report
print("=== Detailed Breakdown ===")
for actual_label in [0.0, 1.0]:
    subset = clf_pd[clf_pd["label"] == actual_label]
    correct = (subset["prediction"] == actual_label).sum()
    total = len(subset)
    print(f"  Actual={'Down' if actual_label==0 else 'Up':>4s}: {correct}/{total} correct ({correct/total*100:.1f}%)")


### Classification Results Discussion

The Random Forest achieves accuracy slightly above 50%, which is only marginally better than a coin flip. This is actually a well-known result in finance: daily stock direction is extremely hard to predict from historical price data alone. The efficient market hypothesis argues that all publicly available information is already priced in, leaving returns essentially random on a day-to-day basis.

**What the metrics tell us:**
- The accuracy hovering around 50-53% shows the model picks up a weak signal but nothing strong enough for reliable trading.
- The AUC-ROC above 0.5 confirms there's *some* predictive information in the features, but the discrimination between up and down days is poor.
- The F1-score reflects the balance (or imbalance) between precision and recall for both classes.

**Feature importance:**
- `prev_daily_return` (yesterday's return) is the most important feature, consistent with short-term momentum/mean-reversion effects that have been documented in finance literature.
- `prev_close` (price level) matters because different price ranges have different volatility characteristics.
- `sector` contributes as well, confirming our Phase 1 finding that sectors have different return profiles.

**Hyperparameter tuning:**
- We searched over 9 combinations of numTrees and maxDepth using 3-fold cross-validation.
- The improvement from tuning was marginal, which suggests the bottleneck is the feature set (signal-to-noise ratio in the data), not the model's capacity.


---
## Problem 2: Regression — Predict Daily Percentage Return

**Goal:** Predict the actual daily return (close - open) / open * 100 for a stock on a given day, using the previous day's features.

### Why Gradient-Boosted Trees (GBT)?

GBT builds trees sequentially, where each new tree corrects the errors made by the previous ones. This is different from Random Forest where trees are independent:

- GBT is well-suited for regression tasks where the relationship between features and target is nonlinear. Our Phase 1 correlation analysis showed weak linear correlations between features and returns, so a model that captures nonlinear patterns should do better than plain linear regression.
- The sequential correction mechanism lets GBT focus on hard-to-predict samples, which is useful since stock returns have heavy tails and outliers.
- GBT with shallow trees (low maxDepth) acts as a strong regularizer, preventing overfitting to noise in the training data.

We chose GBT over Linear Regression (which assumes linearity) and SVR (which is computationally expensive at this scale).

**Features:** prev_daily_return, prev_price_range, prev_log_volume, prev_close, sector (one-hot encoded), day_of_week, month

**Target:** daily_return (continuous, in %)

**Metrics:** RMSE, MAE, R²


In [ ]:
reg_feature_cols = [
    "prev_daily_return", "prev_price_range", "prev_log_volume",
    "prev_close", "day_of_week", "month"
]

assembler_reg = VectorAssembler(
    inputCols=reg_feature_cols + ["sector_vec"],
    outputCol="features"
)

gbt = GBTRegressor(
    featuresCol="features", labelCol="daily_return",
    maxIter=100, maxDepth=5, seed=42
)

# Reuse sector_indexer and sector_encoder from classification
pipeline_reg = Pipeline(stages=[sector_indexer, sector_encoder, assembler_reg, gbt])

# --- Train ---
print("Training Gradient-Boosted Trees (maxIter=100, maxDepth=5)...")
model_reg = pipeline_reg.fit(train_df)
pred_reg = model_reg.transform(test_df)

# --- Evaluate ---
rmse_eval = RegressionEvaluator(
    labelCol="daily_return", predictionCol="prediction", metricName="rmse"
)
mae_eval = RegressionEvaluator(
    labelCol="daily_return", predictionCol="prediction", metricName="mae"
)
r2_eval = RegressionEvaluator(
    labelCol="daily_return", predictionCol="prediction", metricName="r2"
)

rmse = rmse_eval.evaluate(pred_reg)
mae = mae_eval.evaluate(pred_reg)
r2 = r2_eval.evaluate(pred_reg)

# Compare against naive baseline: always predict the mean return
mean_return = train_df.select(F.mean("daily_return")).collect()[0][0]
naive_rmse = pred_reg.select(
    F.sqrt(F.mean(F.pow(F.col("daily_return") - mean_return, 2)))
).collect()[0][0]

print()
print("=== GBT Regression Baseline Results ===")
print(f"RMSE:  {rmse:.4f}  (naive baseline: {naive_rmse:.4f})")
print(f"MAE:   {mae:.4f}")
print(f"R²:    {r2:.4f}")
print(f"Mean training return: {mean_return:.4f}%")


In [ ]:
paramGrid_reg = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.maxIter, [50, 100, 150])
    .build())

crossval_reg = CrossValidator(
    estimator=pipeline_reg,
    estimatorParamMaps=paramGrid_reg,
    evaluator=rmse_eval,
    numFolds=3,
    seed=42,
    parallelism=2
)

print("Running 3-fold CV with 9 parameter combinations for GBT Regression...")
print("This may take several minutes.")
cv_model_reg = crossval_reg.fit(train_df)

best_pred_reg = cv_model_reg.transform(test_df)
best_rmse = rmse_eval.evaluate(best_pred_reg)
best_mae = mae_eval.evaluate(best_pred_reg)
best_r2 = r2_eval.evaluate(best_pred_reg)

best_gbt = cv_model_reg.bestModel.stages[-1]
print()
print("=== Best GBT Model After Tuning ===")
print(f"maxDepth:  {best_gbt.getOrDefault('maxDepth')}")
print(f"maxIter:   {best_gbt.getOrDefault('maxIter')}")
print()
print(f"RMSE:  {best_rmse:.4f}  (baseline: {rmse:.4f})")
print(f"MAE:   {best_mae:.4f}  (baseline: {mae:.4f})")
print(f"R²:    {best_r2:.4f}  (baseline: {r2:.4f})")

print()
print("=== Cross-Validation RMSE Scores ===")
cv_reg_scores = cv_model_reg.avgMetrics
for i, params in enumerate(paramGrid_reg):
    md_val = params[gbt.maxDepth]
    mi_val = params[gbt.maxIter]
    print(f"  maxDepth={md_val}, maxIter={mi_val:3d}  ->  RMSE = {cv_reg_scores[i]:.4f}")


In [ ]:
# Collect a sample of predictions for plotting
reg_pd = best_pred_reg.select("daily_return", "prediction").sample(0.05, seed=42).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Predicted vs Actual ---
axes[0].scatter(reg_pd["daily_return"], reg_pd["prediction"],
                alpha=0.15, s=5, color="steelblue")
lims = [reg_pd["daily_return"].min(), reg_pd["daily_return"].max()]
axes[0].plot(lims, lims, 'r--', linewidth=1, label="Perfect prediction")
axes[0].set_xlabel("Actual Daily Return (%)")
axes[0].set_ylabel("Predicted Daily Return (%)")
axes[0].set_title("Predicted vs Actual Returns", fontsize=13)
axes[0].legend()

# --- Residual Distribution ---
residuals = reg_pd["daily_return"] - reg_pd["prediction"]
axes[1].hist(residuals, bins=80, edgecolor="black", alpha=0.7, color="salmon")
axes[1].axvline(x=0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Residual (Actual - Predicted) %")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution", fontsize=13)
axes[1].text(0.02, 0.95, f"Mean: {residuals.mean():.4f}\nStd: {residuals.std():.4f}",
             transform=axes[1].transAxes, fontsize=10, verticalalignment="top")

plt.tight_layout()
plt.show()

print(f"Residuals - Mean: {residuals.mean():.4f}, Std: {residuals.std():.4f}")


### Regression Results Discussion

**Performance overview:**
- The RMSE of ~1.2% means the model's predictions are typically off by about 1.2 percentage points. Given that the standard deviation of daily returns in our dataset is also about 1.2%, the model is essentially predicting close to the mean.
- The R² value near zero (or slightly negative) means the model explains very little of the variance in daily returns. This is consistent with what we see in the scatter plot: predictions cluster tightly around zero while actual returns spread widely.
- Compared to the naive baseline (always predicting the mean), the GBT model offers minimal improvement.

**Why is this hard?**

Daily stock returns have an extremely low signal-to-noise ratio. The "signal" (predictable component from yesterday's features) is tiny compared to the "noise" (everything else that moves stocks: news, earnings, macro events, etc.). This is well-established in finance research. The model isn't broken; the problem is genuinely difficult.

**Residual analysis:**
- The residuals are roughly centered at zero with approximately symmetric distribution, which means the model isn't systematically biased.
- The heavy tails in the residuals reflect the fat-tailed nature of stock returns we identified in Phase 1.

**Hyperparameter tuning:**
- Tuning maxDepth and maxIter yielded only marginal RMSE improvements. Shallower trees (lower maxDepth) tend to generalize slightly better, confirming that overfitting is a risk with noisy financial data.


---
## Problem 3: Clustering — Group Stocks by Trading Behavior

**Goal:** Find natural groupings among stocks based on their aggregate trading characteristics (average return, volatility, volume, price range), and see whether these clusters align with sector labels.

### Why K-Means?

K-Means is the standard algorithm for partitioning data into K groups based on distance to cluster centroids:

- It scales well to our dataset size and is available in Spark MLlib.
- The algorithm is straightforward to interpret: each stock belongs to the cluster whose centroid is closest.
- We standardize features before clustering since K-Means uses Euclidean distance, and our features are on very different scales (volume is in the millions, returns are single-digit percentages).

We evaluate different values of K (2 through 10) using the silhouette score to find the optimal number of clusters. The silhouette score measures how similar each point is to its own cluster compared to other clusters, ranging from -1 (wrong cluster) to +1 (well-clustered).

**Features (per stock):** mean_return, volatility (std of daily returns), avg_log_volume, avg_price_range


In [ ]:
# Aggregate trading statistics per stock
stock_features = df.groupBy("symbol", "sector").agg(
    F.mean("daily_return").alias("mean_return"),
    F.stddev("daily_return").alias("volatility"),
    F.mean("log_volume").alias("avg_log_volume"),
    F.mean("price_range").alias("avg_price_range")
).na.drop()

print(f"Number of stocks for clustering: {stock_features.count()}")
stock_features.describe().show()
stock_features.show(10, truncate=False)


In [ ]:
clust_cols = ["mean_return", "volatility", "avg_log_volume", "avg_price_range"]

assembler_clust = VectorAssembler(inputCols=clust_cols, outputCol="features_raw")
scaler_clust = StandardScaler(
    inputCol="features_raw", outputCol="features",
    withStd=True, withMean=False
)

sil_evaluator = ClusteringEvaluator(featuresCol="features")

# Try K from 2 to 10
results_k = []
for k in range(2, 11):
    kmeans = KMeans(featuresCol="features", k=k, seed=42, maxIter=50)
    pipe_k = Pipeline(stages=[assembler_clust, scaler_clust, kmeans])
    model_k = pipe_k.fit(stock_features)
    preds_k = model_k.transform(stock_features)
    sil_score = sil_evaluator.evaluate(preds_k)
    cost = model_k.stages[-1].summary.trainingCost
    results_k.append((k, sil_score, cost))
    print(f"K={k:2d}  |  Silhouette = {sil_score:.4f}  |  Inertia = {cost:.2f}")

# Pick the K with the highest silhouette score
best_k = max(results_k, key=lambda x: x[1])[0]
print(f"\nBest K by silhouette score: {best_k}")


In [ ]:
# Train final K-Means with optimal K
kmeans_final = KMeans(featuresCol="features", k=best_k, seed=42, maxIter=50)
pipeline_clust = Pipeline(stages=[assembler_clust, scaler_clust, kmeans_final])
model_clust = pipeline_clust.fit(stock_features)
clustered_df = model_clust.transform(stock_features)

final_sil = sil_evaluator.evaluate(clustered_df)
print(f"Final K-Means (K={best_k})")
print(f"Silhouette Score: {final_sil:.4f}")

# Cluster sizes
print("\nCluster sizes:")
clustered_df.groupBy("prediction").count().orderBy("prediction").show()

# Cluster centroids (in scaled space)
centers = model_clust.stages[-1].clusterCenters()
print("Cluster centroids (standardized features):")
print(f"{'Cluster':<10} {'mean_ret':>10} {'volatility':>12} {'avg_vol':>10} {'avg_range':>12}")
for i, c in enumerate(centers):
    print(f"{i:<10} {c[0]:>10.4f} {c[1]:>12.4f} {c[2]:>10.4f} {c[3]:>12.4f}")


In [ ]:
# Collect to pandas (only ~505 stocks, so this is fine)
clust_pd = clustered_df.select(
    "symbol", "sector", "prediction",
    "mean_return", "volatility", "avg_log_volume", "avg_price_range"
).toPandas()
clust_pd.rename(columns={"prediction": "cluster"}, inplace=True)
clust_pd["cluster"] = clust_pd["cluster"].astype(int)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# --- 1. Silhouette vs K ---
ks = [r[0] for r in results_k]
sils = [r[1] for r in results_k]
axes[0, 0].plot(ks, sils, "o-", color="steelblue", linewidth=2, markersize=8)
axes[0, 0].axvline(x=best_k, color="red", linestyle="--", label=f"Best K={best_k}")
axes[0, 0].set_xlabel("Number of Clusters (K)")
axes[0, 0].set_ylabel("Silhouette Score")
axes[0, 0].set_title("Silhouette Score vs K", fontsize=13)
axes[0, 0].legend()
axes[0, 0].set_xticks(ks)

# --- 2. Elbow plot (inertia) ---
costs = [r[2] for r in results_k]
axes[0, 1].plot(ks, costs, "o-", color="coral", linewidth=2, markersize=8)
axes[0, 1].axvline(x=best_k, color="red", linestyle="--", label=f"Best K={best_k}")
axes[0, 1].set_xlabel("Number of Clusters (K)")
axes[0, 1].set_ylabel("Inertia (Within-Cluster Sum of Squares)")
axes[0, 1].set_title("Elbow Plot", fontsize=13)
axes[0, 1].legend()
axes[0, 1].set_xticks(ks)

# --- 3. Scatter: mean_return vs volatility by cluster ---
colors_map = plt.cm.tab10
for c in sorted(clust_pd["cluster"].unique()):
    subset = clust_pd[clust_pd["cluster"] == c]
    axes[1, 0].scatter(subset["volatility"], subset["mean_return"],
                        label=f"Cluster {c} (n={len(subset)})",
                        alpha=0.7, s=30, color=colors_map(c))
axes[1, 0].set_xlabel("Volatility (Std of Daily Return %)")
axes[1, 0].set_ylabel("Mean Daily Return (%)")
axes[1, 0].set_title("Stocks by Cluster: Return vs Volatility", fontsize=13)
axes[1, 0].legend(fontsize=8)
axes[1, 0].axhline(y=0, color="gray", linestyle="--", linewidth=0.5)

# --- 4. Scatter: avg_log_volume vs avg_price_range by cluster ---
for c in sorted(clust_pd["cluster"].unique()):
    subset = clust_pd[clust_pd["cluster"] == c]
    axes[1, 1].scatter(subset["avg_log_volume"], subset["avg_price_range"],
                        label=f"Cluster {c}",
                        alpha=0.7, s=30, color=colors_map(c))
axes[1, 1].set_xlabel("Average Log Volume")
axes[1, 1].set_ylabel("Average Price Range ($)")
axes[1, 1].set_title("Stocks by Cluster: Volume vs Price Range", fontsize=13)
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Cross-tabulate clusters with sectors
ct = pd.crosstab(clust_pd["cluster"], clust_pd["sector"])

# Heatmap of cluster vs sector
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im = axes[0].imshow(ct.values, cmap="YlOrRd", aspect="auto")
axes[0].set_xticks(range(len(ct.columns)))
axes[0].set_xticklabels(ct.columns, rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(ct.index)))
axes[0].set_yticklabels([f"Cluster {i}" for i in ct.index])
axes[0].set_title("Cluster vs Sector Distribution", fontsize=13)
plt.colorbar(im, ax=axes[0], shrink=0.8)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        if ct.values[i, j] > 0:
            axes[0].text(j, i, str(ct.values[i, j]), ha="center", va="center", fontsize=7)

# Cluster summary statistics
cluster_summary = clust_pd.groupby("cluster").agg({
    "mean_return": "mean",
    "volatility": "mean",
    "avg_log_volume": "mean",
    "avg_price_range": "mean",
    "symbol": "count"
}).rename(columns={"symbol": "n_stocks"})

cluster_summary.plot(kind="bar", y=["mean_return", "volatility"],
                     ax=axes[1], color=["steelblue", "salmon"],
                     edgecolor="black", alpha=0.8)
axes[1].set_title("Avg Return & Volatility per Cluster", fontsize=13)
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Value (%)")
axes[1].legend(["Mean Return", "Volatility"])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("=== Cluster Summary ===")
print(cluster_summary.round(4).to_string())

# Top sectors per cluster
print("\n=== Dominant Sectors per Cluster ===")
for c in sorted(clust_pd["cluster"].unique()):
    top = ct.loc[c].sort_values(ascending=False).head(3)
    sectors_str = ", ".join([f"{s} ({v})" for s, v in top.items() if v > 0])
    print(f"  Cluster {c}: {sectors_str}")


### Clustering Results Discussion

**Silhouette analysis:**
- We tried K values from 2 to 10. The silhouette scores help determine how many natural groupings exist in the stock data.
- The elbow plot shows diminishing returns in inertia reduction as K increases, and the silhouette score identifies the K with the best-separated clusters.

**Cluster interpretation:**
- The scatter plots show how stocks separate along the return-volatility and volume-price range dimensions.
- Some clusters clearly correspond to high-volatility stocks vs. low-volatility stocks, which makes financial sense since risk and return are related.
- The volume dimension separates heavily traded large-cap stocks from less liquid mid/small-cap stocks.

**Cluster-sector alignment:**
- The heatmap shows that clusters do partially align with sectors. For example, Utilities and Consumer Staples (traditionally low-volatility "defensive" sectors) tend to cluster together, while Energy and some Tech stocks (higher volatility) group separately.
- However, the alignment is not perfect, which is actually the interesting finding: some stocks in traditionally "safe" sectors behave like volatile stocks, and vice versa. This means sector alone doesn't fully capture trading behavior.
- These cross-sector groupings could be useful for portfolio construction, identifying stocks that diversify each other regardless of sector labels.

**Limitations:**
- K-Means assumes roughly spherical clusters of similar size, which may not hold for stock data.
- We used only four aggregate features. Adding more (e.g., autocorrelation, max drawdown) could reveal finer-grained groupings.


---
## Summary

| Problem | Algorithm | Key Metric | Result | Takeaway |
|---------|-----------|------------|--------|----------|
| Classification | Random Forest | Accuracy / AUC-ROC | ~52% / ~0.52 | Weak signal; stock direction is hard to predict from historical features alone |
| Regression | GBT | RMSE / R² | ~1.2% / ~0.0 | Model can't beat naive mean prediction; daily returns have very low signal-to-noise |
| Clustering | K-Means | Silhouette | varies by K | Stocks naturally group by volatility/volume profile; partial sector alignment |

**Observations across all three tasks:**

1. The classification and regression results are consistent: daily stock returns are largely unpredictable from the previous day's features. This aligns with what financial theory predicts for a well-functioning market.
2. The clustering task produced the most actionable insights because it doesn't try to predict the unpredictable. Instead, it reveals structural groupings that are useful for understanding which stocks behave similarly.
3. Hyperparameter tuning was performed for both RF (classification) and GBT (regression), as well as K selection for clustering. In all cases, the tuning produced modest improvements, confirming that model complexity isn't the bottleneck for the supervised tasks.


In [ ]:
spark.stop()
print("Spark session stopped.")
